# L7.5 · SmolVLA: el experto de acciones

| Notebook | Qué abre |
|---|---|
| L7.1 | El mapa: qué entra, qué sale, dónde están los 450 M de parámetros |
| L7.2 | La imagen: de la foto a 64 tokens |
| L7.3 | La fila del VLM: palabras, estado y quién puede mirar a quién |
| L7.4 | Adentro del VLM: qué miran las palabras, por qué 16 de 32 capas, qué queda guardado |
| `06` (flujo) | Entrenarlo con nuestros datos |
| **L7.5** | El experto de acciones: la brújula de L6 con ojos |
| L7.6 | Las 10 pasadas: un bloque de verdad pasando de ruido a movimiento |
| L7.7 | El entrenamiento por dentro |
| `07` (flujo) | Evaluación contra ACT, en los mismos episodios |

Desde acá usamos **el modelo entrenado con nuestros datos** (`06`): el
experto del modelo base no sabe mover un G1 (L7.1).

In [ ]:
import torch  # antes que lerobot: evita un import circular entre torchvision y torch._dynamo

import matplotlib.pyplot as plt
import numpy as np
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.policies.smolvla.modeling_smolvla import make_att_2d_masks

from toolkit.paths import CHECKPOINTS_DIR, DATASETS_DIR
from toolkit.policy import smolvla_internals as vla
from toolkit.policy.smolvla_g1_base import load_g1_policy

plt.rcParams["figure.dpi"] = 110
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)

TRAINED = CHECKPOINTS_DIR / "smolvla_g1_v1" / "checkpoints" / "last" / "pretrained_model"
TRAIN = DATASETS_DIR / "local" / "pick_mameluco_act_v1_train"
train_meta = LeRobotDataset("local/pick_mameluco_act_v1_train", root=TRAIN).meta
heldout = LeRobotDataset("local/pick_mameluco_act_v1_heldout", root=DATASETS_DIR / "local" / "pick_mameluco_act_v1_heldout",
                         delta_timestamps={"action": [i / 30 for i in range(50)]})
policy, preprocess, postprocess = load_g1_policy(TRAINED, train_meta.stats, device)
model = policy.model
expert = model.vlm_with_expert.lm_expert

# Un instante de un episodio que el modelo nunca vio al entrenar.
frame = heldout[int(heldout.meta.episodes["dataset_from_index"][1]) + 120]
obs = vla.prepare_observation(policy, preprocess(dict(frame)))
with torch.no_grad():
    prefix = vla.embed_prefix(model, obs)
    kv_cache = vla.run_prefix(model, prefix)
image = obs.images[0]


def show(ax, img):
    ax.imshow(((img[0].permute(1, 2, 0).float().cpu() + 1) / 2).clamp(0, 1).numpy())
    ax.axis("off")

## 1. El problema

Del VLM nos quedó la caché: las etiquetas y contenidos de los 113 tokens de la
fila, en cada una de sus 16 capas (L7.4). Ahora hay que generar un bloque de 50
acciones con flow matching: **la brújula de L6**, pero con dos diferencias.

- El "punto" ya no tiene 2 números: es un bloque de 50 acciones × 32 números
  (las 26 articulaciones, rellenadas hasta 32).
- La brújula no mira sólo (punto, momento): tiene que mirar **la escena**, que
  está en la caché.

El experto de acciones es esa brújula: un transformer de 16 capas, más angosto
que el VLM (720 en vez de 960), que en cada pasada recibe el bloque actual y el
momento, consulta la caché y devuelve la velocidad.

Una aclaración de convención: LeRobot escribe el tiempo al revés que la serie
L6. Acá `t = 1` es ruido puro y `t = 0` es el bloque terminado.

## 2. La entrada: el bloque ruidoso y el momento

En L6.2, la brújula recibía el punto y el momento pegados como 3 números. Acá
el bloque se vuelve **50 tokens**, uno por acción: una matriz de 32 × 720
convierte cada acción en un vector.

El momento `t` es un solo número, y una red aprende mal de un número suelto:
`t = 0,50` y `t = 0,52` quedan casi iguales. El truco es convertirlo en **muchas
ondas de distinto período** (senos y cosenos): las ondas rápidas distinguen
momentos cercanos, y las lentas dicen "más o menos dónde estamos".

In [ ]:
times = torch.linspace(0, 1, 200, device=device)
waves = vla.time_embedding(model, times).float().cpu()
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.imshow(waves.T, aspect="auto", cmap="RdBu", extent=(0, 1, waves.shape[1], 0))
ax.set_xlabel("momento t (1 = ruido, 0 = bloque terminado)")
ax.set_ylabel("número del vector (720)")
ax.set_title("el momento convertido en 720 ondas: cada columna es el vector de un t")
plt.tight_layout()
print("matriz de acciones:", tuple(model.action_in_proj.weight.T.shape),
      "· MLP que mezcla acción y momento:", tuple(model.action_time_mlp_in.weight.T.shape), "→",
      tuple(model.action_time_mlp_out.weight.T.shape))

Después, en cada uno de los 50 tokens se pega el vector de la acción con el
del momento (720 + 720) y un MLP chico los mezcla en un vector de 720. Los 50
casilleros reciben el mismo momento: todo el bloque avanza junto.

In [ ]:
noise = torch.randn(1, model.config.chunk_size, model.config.max_action_dim, device=device)
t_start = torch.ones(1, device=device)
with torch.no_grad():
    suffix, suffix_pad, suffix_att = vla.embed_suffix(model, noise, t_start)
print("bloque de ruido:", tuple(noise.shape), "→ fila del experto:", tuple(suffix.shape))

## 3. Las capas del experto: alternadas

El experto tiene 16 capas, una por cada capa del VLM, y cada una consulta la
caché **de la misma capa** del VLM. Pero no todas consultan igual:

- **Capas pares, atención conjunta** (como π0): las acciones calculan sus
  preguntas, etiquetas y contenidos, se juntan con los guardados del VLM y se
  hace una sola atención. Cada acción ve la escena **y a las acciones
  anteriores del bloque**.
- **Capas impares, atención cruzada**: las acciones sólo **leen** la escena.
  No se ven entre ellas. Como lo guardado del VLM tiene otro ancho, una matriz
  lo traduce al ancho del experto antes de usarlo.

In [ ]:
for index in [0, 1]:
    attention = expert.layers[index].self_attn
    kind = "conjunta" if index % model.config.self_attn_every_n_layers == 0 else "cruzada"
    print(f"capa {index} ({kind}): Q {tuple(attention.q_proj.weight.T.shape)} · K {tuple(attention.k_proj.weight.T.shape)} "
          f"· V {tuple(attention.v_proj.weight.T.shape)} · MLP {tuple(expert.layers[index].mlp.up_proj.weight.T.shape)}")
print("tipos:", " ".join("C" if i % 2 == 0 else "x" for i in range(16)), " (C = conjunta, x = cruzada)")

paper = {"sólo cruzada": 79.0, "sólo conjunta": 74.5, "alternadas\n(SmolVLA)": 85.5}
fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.bar(list(paper), list(paper.values()), color=["lightgray", "lightgray", "tab:red"])
for bar, value in zip(bars, paper.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.3, f"{value}", ha="center")
ax.set_ylim(70, 88)
ax.set_ylabel("éxito promedio (%)")
ax.set_title("cómo conectar experto y VLM (paper, tabla 6, LIBERO)")
plt.tight_layout()

*(Interpretación pendiente: se escribe con el modelo final del entrenamiento.)*

## 4. La máscara completa

Juntando la fila del VLM (113) y la del experto (50), la regla de L7.3 se
extiende: cada acción lleva un 1, así que **cada acción puede mirar a las
anteriores y no a las siguientes**, y nada del VLM puede mirar a las acciones.
En las capas cruzadas, las acciones sólo ven la parte del VLM.

In [ ]:
pad = torch.cat([prefix.pad_masks, suffix_pad], dim=1)
att = torch.cat([prefix.att_masks, suffix_att], dim=1)
full = make_att_2d_masks(pad, att)[0].cpu().numpy()
n_prefix = prefix.embs.shape[1]

fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [1.3, 1]})
axes[0].imshow(full, cmap="Greys", interpolation="nearest")
axes[0].axhline(n_prefix - 0.5, color="tab:red")
axes[0].axvline(n_prefix - 0.5, color="tab:red")
axes[0].set_title("las dos filas juntas (negro = puede mirar)")
axes[0].set_xlabel("a quién mira")
axes[0].set_ylabel("quién mira")
axes[0].text(n_prefix + 25, n_prefix - 8, "acciones", color="tab:red", ha="center")
axes[1].imshow(full[n_prefix:, n_prefix:], cmap="Greys", interpolation="nearest")
axes[1].set_title("zoom: las 50 acciones entre ellas (capas conjuntas)")
axes[1].set_xlabel("acción a la que mira")
axes[1].set_ylabel("acción que mira")
plt.tight_layout()

El triángulo es la regla "cada acción mira sólo las anteriores". El paper
probó tres variantes (tabla 7): que las acciones no se miren entre ellas, que
se miren en las dos direcciones, o sólo hacia atrás. Hacia atrás fue la mejor;
mirarse en las dos direcciones empeoró. Y sorprendentemente, que no se miren
para nada anduvo casi igual de bien: la escena sola ya dice mucho.

## 5. Dónde mira cada acción

Generamos un bloque entero (10 pasadas) capturando la atención. En cada pasada
hay 16 llamadas, una por capa. Repartimos la atención de las acciones entre la
imagen, las palabras, el estado y las acciones anteriores, en la **primera**
pasada (`t = 1`, puro ruido) y en la **última** (`t = 0,1`, casi terminado):

In [ ]:
with torch.no_grad(), vla.capture_attention(model) as log:
    path = vla.sample_with_path(model, obs, noise)
layers_n = model.vlm_with_expert.num_vlm_layers
step_probs = [log.probs[layers_n * (1 + s): layers_n * (2 + s)] for s in range(model.config.num_steps)]
seg = prefix.segments
real_text = slice(seg["texto"].start, seg["texto"].start + int(obs.lang_masks[0].sum()))


def split_actions(probs):
    """Atención promedio de las 50 acciones (y las 15 cabezas) repartida por destino."""
    a = probs[0].mean(0).mean(0)  # (destinos,)
    parts = {"imagen": a[seg["imagen"]].sum(), "palabras": a[real_text].sum(), "estado": a[seg["estado"]].sum()}
    parts["acciones"] = a[n_prefix:].sum() if a.shape[0] > n_prefix else torch.tensor(0.0)
    return parts


fig, axes = plt.subplots(1, 2, figsize=(14, 3.8), sharey=True)
colors = {"imagen": "tab:blue", "palabras": "tab:green", "estado": "tab:orange", "acciones": "tab:red"}
for ax, s, title in [(axes[0], 0, "primera pasada (t = 1, ruido)"), (axes[1], model.config.num_steps - 1, "última pasada (t = 0,1)")]:
    bottom = np.zeros(layers_n)
    for name in colors:
        values = np.array([split_actions(p)[name].item() for p in step_probs[s]])
        ax.bar(range(layers_n), values, bottom=bottom, color=colors[name], label=name)
        bottom += values
    ax.set_xlabel("capa del experto (pares: conjunta · impares: cruzada)")
    ax.set_title(title)
axes[0].set_ylabel("fracción de la atención")
axes[1].legend(fontsize=8, loc="lower right")
plt.tight_layout()

*(Interpretación pendiente: se escribe con el modelo final del entrenamiento.)*

Y sobre la foto: a qué tokens de imagen miran las acciones en algunas capas
cruzadas de la última pasada (promedio de acciones y cabezas, sin el sumidero
de L7.4, marcado con ×):

In [ ]:
sink = 63
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
for ax, layer_index in zip(axes, [1, 5, 9, 15]):
    to_image = step_probs[-1][layer_index][0].mean(0).mean(0)[seg["imagen"]].clone()
    share = to_image.sum().item()
    to_image[sink] = float("nan")
    to_image = (to_image / np.nanmax(to_image.numpy())).reshape(8, 8).numpy()
    show(ax, image)
    ax.imshow(np.kron(to_image, np.ones((64, 64))), cmap="inferno", alpha=0.6, extent=(0, 512, 512, 0))
    ax.plot((sink % 8) * 64 + 32, (sink // 8) * 64 + 32, marker="x", color="white", markersize=10)
    ax.set_title(f"capa {layer_index} · {share:.0%} a la imagen", fontsize=9)
plt.tight_layout()

*(Interpretación pendiente: se escribe con el modelo final del entrenamiento.)*

## 6. La salida: una velocidad para cada número del bloque

Después de las 16 capas, una matriz de 720 × 32 convierte cada uno de los 50
tokens en la velocidad de su acción: hacia dónde mover cada uno de los 32
números. Es exactamente lo que devolvía la brújula de L6, pero en 1600
dimensiones.

¿Y las 6 dimensiones de relleno (de la 27 a la 32)? Ahí la acción "verdadera"
es siempre 0, así que la velocidad correcta tiene una forma fija: con la
convención de LeRobot, `x_t = t · ruido` y la velocidad es `ruido = x_t / t`.
Miramos qué devuelve el modelo ahí, en la mitad del camino:

In [ ]:
t_half = torch.full((1,), 0.5, device=device)
x_half = path[5]  # el bloque después de 5 pasadas: t = 0,5
with torch.no_grad():
    velocity = vla.velocity(model, prefix, kv_cache, x_half, t_half)
padded_dims = slice(26, 32)
expected = (x_half / 0.5)[..., padded_dims].flatten().cpu()
predicted = velocity[..., padded_dims].flatten().cpu()
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.scatter(expected, predicted, s=6)
lim = float(expected.abs().max())
ax.plot([-lim, lim], [-lim, lim], color="gray", ls="--")
ax.set_xlabel("velocidad correcta (x_t / t)")
ax.set_ylabel("velocidad del modelo")
ax.set_title("las 6 dimensiones de relleno")
plt.tight_layout()
print(f"error típico en el relleno: {(predicted - expected).abs().median():.3f}")

*(Interpretación pendiente: se escribe con el modelo final del entrenamiento.)*

## Lo que nos llevamos

- El experto es la brújula de L6 con ojos: recibe el bloque ruidoso (50
  tokens) y el momento (en forma de ondas), y devuelve una velocidad por número.
- Sus 16 capas leen la caché del VLM, cada una la de su misma capa, y
  alternan: conjuntas (escena + acciones anteriores) y cruzadas (sólo escena).
- Cada acción mira las anteriores del bloque, nunca las siguientes.
- Las dimensiones de relleno también se entrenan, aunque no sirven para nada:
  es uno de los problemas de LeRobot 0.4.1 que vemos en L7.7.

## Referencias

- Shukor et al., [SmolVLA](https://arxiv.org/abs/2506.01844) (2025), sección
  3.1 y tablas 6, 7 y 9.
- Black et al., [π0](https://arxiv.org/abs/2410.24164) (2024): la atención
  conjunta y el momento como ondas.
- Vaswani et al., [Attention Is All You Need](https://arxiv.org/abs/1706.03762)
  (2017): las ondas de senos y cosenos para posiciones.